<a href="https://colab.research.google.com/github/Kuper994/Software-Project/blob/sp27/large_scale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Large-Scale C Project — Hands-On Lecture

This notebook is designed as an **interactive coding workshop**, not a slide deck.

The rhythm is:

**short explanation → predict → code/change something → run → inspect the result → discuss**

You will repeatedly be asked to make a change before seeing the answer. Do not skip the exercises: the mistakes and observations are part of the lecture.

## 0. Setup: check the tools

We will use a small command-line C project. The notebook will create all source files locally, so there is no dependency on GitHub or on a particular starter repository.

If one of these commands is unavailable on your machine, tell the instructor before continuing.


In [ ]:
import shutil
import subprocess
from pathlib import Path

for tool in ["gcc", "make", "git"]:
    print(f"{tool:>5}: {shutil.which(tool) or 'NOT FOUND'}")

print("Python:", shutil.which("python") or shutil.which("python3"))


## 1. Start with one C file

Before introducing modules, let's make sure we can compile and run a tiny program.

**Do not just read the code. Run it, then change the output and run it again.**


In [ ]:
from pathlib import Path

WORKDIR = Path.cwd() / "large_scale_lecture_project"
if WORKDIR.exists():
    import shutil
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(exist_ok=True)

hello = WORKDIR / "hello.c"
hello.write_text(r'''
#include <stdio.h>

int main(void)
{
    printf("Hello from C!\n");
    return 0;
}
'''.lstrip())

print(hello.read_text())


In [ ]:
result = subprocess.run(
    ["gcc", "-Wall", "-Wextra", "-std=c11", "-o", str(WORKDIR / "hello"), str(hello)],
    text=True, capture_output=True
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0


In [ ]:
result = subprocess.run([str(WORKDIR / "hello")], text=True, capture_output=True)
print(result.stdout)


### Exercise: Predict the compiler command

Before running anything, write the exact `gcc` command you would use to compile `hello.c` with warnings enabled and explain what `-o` changes. Then run the check.

In [ ]:
# Write your answer as a comment, then run the command.
# gcc ...

### Check your work

In [ ]:
print("The expected shape is: gcc -Wall -Wextra -std=c11 -o <output> <source>.")

### Your turn — make a change

Change the program so that it prints **two different lines**. Re-run the compilation cell and then run the executable.

**Question:** what has to happen when `hello.c` changes? Why can't we simply run the old executable?


# 2. C modules: split the program

A large program should not become one enormous `.c` file.

We will build a tiny calculator with:
- `main.c` — user-facing program;
- `math_utils.c` / `math_utils.h` — arithmetic functions;
- `string_utils.c` / `string_utils.h` — formatting functions.

The `.h` files contain **declarations**; the `.c` files contain **definitions**.


In [ ]:
# Create the initial multi-file project
(WORKDIR / "src").mkdir(exist_ok=True)

(WORKDIR / "src" / "math_utils.h").write_text(r'''
#ifndef MATH_UTILS_H
#define MATH_UTILS_H

int add(int a, int b);
int multiply(int a, int b);

#endif
'''.lstrip())

(WORKDIR / "src" / "math_utils.c").write_text(r'''
#include "math_utils.h"

int add(int a, int b)
{
    return a + b;
}

int multiply(int a, int b)
{
    return a * b;
}
'''.lstrip())

(WORKDIR / "src" / "string_utils.h").write_text(r'''
#ifndef STRING_UTILS_H
#define STRING_UTILS_H

void print_result(const char *operation, int value);

#endif
'''.lstrip())

(WORKDIR / "src" / "string_utils.c").write_text(r'''
#include <stdio.h>
#include "string_utils.h"

void print_result(const char *operation, int value)
{
    printf("%s = %d\n", operation, value);
}
'''.lstrip())

(WORKDIR / "src" / "main.c").write_text(r'''
#include "math_utils.h"
#include "string_utils.h"

int main(void)
{
    print_result("2 + 3", add(2, 3));
    print_result("4 * 5", multiply(4, 5));
    return 0;
}
'''.lstrip())

print("\n".join(p.name for p in sorted((WORKDIR / "src").iterdir())))


### Predict before running

Which files does `main.c` need to know about?

1. Does `main.c` need the **implementation** of `add()`?
2. Does `main.c` need the **declaration** of `add()`?
3. Which files need to include `math_utils.h`?
4. If `math_utils.c` changes, should `main.c` be recompiled?

Write down your answers before running the next cell.


In [ ]:
# Compile and link all three source files in one command.
result = subprocess.run(
    ["gcc", "-Wall", "-Wextra", "-std=c11",
     "-o", str(WORKDIR / "calculator"),
     str(WORKDIR / "src" / "main.c"),
     str(WORKDIR / "src" / "math_utils.c"),
     str(WORKDIR / "src" / "string_utils.c")],
    text=True, capture_output=True
)
print(result.stderr)
assert result.returncode == 0

run = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True)
print(run.stdout)


### Exercise: Identify compile-time vs link-time errors

Make two predictions: (1) what happens if a function is declared in the header but has no implementation, and (2) what happens if the declaration is missing entirely. Test both cases by editing the project temporarily.

In [ ]:
# TODO: use Path(...).read_text()/write_text() to make one controlled change.
# Restore the project after observing the error.

### Check your work

In [ ]:
print("Look for the distinction: compiler errors are about translation/type information; linker errors are about missing definitions.")

## 2.1 Compile → link: see the intermediate files

The previous command hides an important process. Compilation and linking are separate steps.

Let's create object files explicitly.


In [ ]:
for source in ["main.c", "math_utils.c", "string_utils.c"]:
    result = subprocess.run(
        ["gcc", "-Wall", "-Wextra", "-std=c11", "-c", str(WORKDIR / "src" / source),
         "-o", str(WORKDIR / "src" / source.replace(".c", ".o"))],
        text=True, capture_output=True
    )
    print(source, "->", result.returncode)
    if result.stderr:
        print(result.stderr)

print("Object files:", [p.name for p in (WORKDIR / "src").glob("*.o")])


In [ ]:
objects = [str(p) for p in sorted((WORKDIR / "src").glob("*.o"))]
result = subprocess.run(
    ["gcc", "-o", str(WORKDIR / "calculator_from_objects"), *objects],
    text=True, capture_output=True
)
print(result.stderr)
assert result.returncode == 0
print(subprocess.run([str(WORKDIR / "calculator_from_objects")], text=True, capture_output=True).stdout)


### Exercise: Inspect the object files

Use `nm` to inspect `math_utils.o`. Find the symbols for `add` and `multiply`. Which symbols are defined here, and which are merely referenced?

In [ ]:
import subprocess
result = subprocess.run(["nm", str(WORKDIR / "src" / "math_utils.o")], text=True, capture_output=True)
print(result.stdout)

### Check your work

In [ ]:
assert "add" in result.stdout
assert "multiply" in result.stdout
print("Good: nm lets you inspect symbols without reading the machine code.")

### Mini-exercise — break the interface

Temporarily change the declaration in `math_utils.h` from:

```c
int add(int a, int b);
```

to an incorrect declaration such as:

```c
int add(int a);
```

Compile again.

**Observe:** the compiler can detect an interface mismatch because the caller and the declaration disagree.

Restore the correct declaration before continuing.


# 3. Dependencies: what actually needs rebuilding?

Now we get to the reason large projects need build systems.

Imagine the project contains:

```text
main.c
math_utils.c
math_utils.h
string_utils.c
string_utils.h
```

Ask the key question:

> If one file changes, which object files become stale?


### Dependency graph

A useful mental model is:

```text
math_utils.h ──> math_utils.c ──> math_utils.o ──┐
       │                                           │
       └────────> main.c ────────> main.o ────────┼──> calculator
                                                   │
string_utils.h ─> string_utils.c ─> string_utils.o┘
       │
       └────────> main.c
```

Notice that a `.c` file is not normally a prerequisite of `main.o` merely because `main.o` eventually needs code from it. `main.o` is compiled from `main.c`; the **header** describes the interface used while compiling `main.c`.


### Exercise: Draw the dependency graph yourself

On paper, draw arrows for every dependency needed to rebuild `calculator` after (a) `main.c` changes, (b) `math_utils.c` changes, and (c) `math_utils.h` changes. Then compare with the checker.

In [ ]:
# Write your prediction here:
# main.c -> ...
# math_utils.c -> ...
# math_utils.h -> ...

### Check your work

In [ ]:
print("Key idea: a header change can invalidate every object that includes it, even when the .c implementation did not change.")

# 4. Make: let the computer manage the dependency graph

Instead of remembering all the `gcc` commands, we can describe the dependency graph once.

First, let's write a minimal `Makefile`.


In [ ]:
makefile = WORKDIR / "Makefile"
makefile.write_text(r'''
CC = gcc
CFLAGS = -Wall -Wextra -std=c11

calculator: main.o math_utils.o string_utils.o
	$(CC) -o calculator main.o math_utils.o string_utils.o

main.o: src/main.c src/math_utils.h src/string_utils.h
	$(CC) $(CFLAGS) -c src/main.c -o main.o

math_utils.o: src/math_utils.c src/math_utils.h
	$(CC) $(CFLAGS) -c src/math_utils.c -o math_utils.o

string_utils.o: src/string_utils.c src/string_utils.h
	$(CC) $(CFLAGS) -c src/string_utils.c -o string_utils.o

clean:
	rm -f calculator *.o
'''.lstrip())

print(makefile.read_text())


In [ ]:
def run_make(*args):
    result = subprocess.run(["make", *args], cwd=WORKDIR, text=True, capture_output=True)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    return result

result = run_make()
assert result.returncode == 0


### Observe incremental compilation

Run `make` **again** without changing anything.

What does Make do differently this time?


In [ ]:
run_make()


### Exercise: Makefile detective

Run `make` twice. Then touch only `src/string_utils.c` and run `make` again. Record exactly which `.o` files and the executable are rebuilt.

In [ ]:
import os, subprocess, time
source = WORKDIR / "src" / "string_utils.c"
os.utime(source, None)
result = subprocess.run(["make"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)

### Check your work

In [ ]:
assert "string_utils.o" in result.stdout
print("The important observation is that Make rebuilds only the affected object, then relinks the executable.")

### Your turn — change only one implementation

Change the return value of `multiply()` from `a * b` to `a * b + 1`.

Before running `make`, predict which commands should run:
- `main.c` → `main.o`?
- `math_utils.c` → `math_utils.o`?
- `string_utils.c` → `string_utils.o`?
- link the final executable?

Then make the change and run `make`.


In [ ]:
path = WORKDIR / "src" / "math_utils.c"
text = path.read_text()
text = text.replace("return a * b;", "return a * b + 1;")
path.write_text(text)

run_make()


### Your turn — change a header

Now change the declaration of `multiply()` in `math_utils.h` **without changing its meaning**, for example by changing whitespace or adding a comment.

Run `make`.

**Why should `main.o` be rebuilt even though `main.c` did not change?**


In [ ]:
path = WORKDIR / "src" / "math_utils.h"
text = path.read_text()
text = text.replace("int multiply(int a, int b);", "int multiply(int a, int b); /* multiplication */")
path.write_text(text)

run_make()


### Exercise: Repair an incomplete dependency rule

The broken rule lists only `main.c`. Fix it so a change to either header forces `main.o` to rebuild. Do not change the command itself.

In [ ]:
makefile = WORKDIR / "Makefile"
text = makefile.read_text()
# TODO: locate the main.o rule and add the two header prerequisites.
makefile.write_text(text)
print("Edit the Makefile above, then run make in the next cell.")

### Check your work

In [ ]:
result = subprocess.run(["make", "-n", "main.o"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)
print("Check that the rule now mentions both header files.")

## 4.1 A common Makefile mistake

Suppose someone writes:

```make
main.o: main.c
```

even though `main.c` includes `math_utils.h`.

Then `make` has no way to know that changing `math_utils.h` makes `main.o` stale.

**Fix the dependency list below.**


In [ ]:
broken_rule = "main.o: src/main.c"
correct_rule = "main.o: src/main.c src/math_utils.h src/string_utils.h"

print("Broken: ", broken_rule)
print("Correct:", correct_rule)
assert "math_utils.h" in correct_rule
assert "string_utils.h" in correct_rule


## 4.2 Cleaner Makefiles: automatic header dependencies

For real projects, manually listing every header can become tedious.

A common GCC approach is to generate `.d` dependency files using `-MMD -MP` and include them from the Makefile.

We will build a second version that demonstrates this pattern.


In [ ]:
auto_makefile = WORKDIR / "Makefile.auto"
auto_makefile.write_text(r'''
CC = gcc
CFLAGS = -Wall -Wextra -std=c11 -MMD -MP

TARGET = calculator
OBJECTS = main.o math_utils.o string_utils.o
DEPS = $(OBJECTS:.o=.d)

$(TARGET): $(OBJECTS)
	$(CC) -o $@ $^

%.o: src/%.c
	$(CC) $(CFLAGS) -c $< -o $@

clean:
	rm -f $(TARGET) $(OBJECTS) $(DEPS)

-include $(DEPS)
'''.lstrip())

print(auto_makefile.read_text())


### Exercise

Copy the automatic-dependency Makefile to `Makefile`, run `make clean`, then `make`.

Afterward, inspect the generated `.d` files. They are Make-readable descriptions of which headers each object file depends on.


In [ ]:
# Switch to the automatic version for the rest of the Make demonstration.
(makefile).write_text(auto_makefile.read_text())
run_make("clean")
run_make()
print("Generated dependency files:", [p.name for p in WORKDIR.glob("*.d")])


### Exercise: Use `make -n` as a safe preview

`make -n` shows commands without executing them. Change a header, then use `make -n` to predict the rebuild. Only after you agree with the output should you run `make`.

In [ ]:
header = WORKDIR / "src" / "string_utils.h"
text = header.read_text()
header.write_text(text + "\n/* lecture exercise */\n")
result = subprocess.run(["make", "-n"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)

### Check your work

In [ ]:
assert "main.o" in result.stdout
print("Good. A dry run is useful when debugging build dependencies.")

# 5. Git: version control by doing, not by slides

We will use a **local Git repository**. This lets everyone practice the workflow without needing a GitHub account or network access.

The basic loop is:

```text
edit → git status → git diff → git add → git diff --staged → git commit
```


In [ ]:
# Configure Git only inside this lecture repository.
subprocess.run(["git", "init"], cwd=WORKDIR, text=True, capture_output=True)
subprocess.run(["git", "branch", "-M", "main"], cwd=WORKDIR, text=True, capture_output=True)
subprocess.run(["git", "config", "user.name", "Lecture Student"], cwd=WORKDIR)
subprocess.run(["git", "config", "user.email", "student@example.com"], cwd=WORKDIR)

result = subprocess.run(["git", "status", "--short"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)


### First commit

Stage the source files and Makefile, inspect what is staged, and commit.

**Try the commands yourself in the terminal or use the next cells.**


In [ ]:
result = subprocess.run(["git", "add", "src", "Makefile"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stderr)
result = subprocess.run(["git", "diff", "--staged", "--stat"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)


In [ ]:
result = subprocess.run(
    ["git", "commit", "-m", "Initial calculator project"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(result.stdout)
print(result.stderr)


### Exercise: Git status prediction

Before running any Git command, predict the state after: edit a file → `git add` → edit the same file again. Which version is in the index? Which version is in the working tree?

In [ ]:
# Write your prediction here:
# index = ...
# working tree = ...

### Check your work

In [ ]:
print("The index contains the version you staged; later edits remain only in the working tree until staged again.")

## 5.1 `git diff` is your microscope

Make a small change, but **do not commit it yet**.

Then inspect:
1. `git status`
2. `git diff`
3. stage the file
4. `git diff --staged`

What is the difference between the last two?


In [ ]:
path = WORKDIR / "src" / "main.c"
text = path.read_text()
text = text.replace('print_result("2 + 3", add(2, 3));', 'print_result("10 + 7", add(10, 7));')
path.write_text(text)

for command in [["git", "status", "--short"], ["git", "diff"]]:
    result = subprocess.run(command, cwd=WORKDIR, text=True, capture_output=True)
    print("$", " ".join(command))
    print(result.stdout)


In [ ]:
subprocess.run(["git", "add", "src/main.c"], cwd=WORKDIR)
result = subprocess.run(["git", "diff", "--staged"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)


### Exercise: Stage selectively

Make two independent changes to two files. Use `git diff` to inspect both, then stage only one file. Verify with `git diff` and `git diff --staged`.

In [ ]:
# TODO: make two small edits in two different source files, then use subprocess.run for git diff/add/status.

### Check your work

In [ ]:
unstaged = subprocess.run(["git", "diff", "--name-only"], cwd=WORKDIR, text=True, capture_output=True).stdout
staged = subprocess.run(["git", "diff", "--staged", "--name-only"], cwd=WORKDIR, text=True, capture_output=True).stdout
print("Unstaged:\n", unstaged)
print("Staged:\n", staged)
print("You should see different files in the two lists.")

### Mini-exercise — commit a meaningful change

Commit the change with a message that describes **what changed**, not just “update code”.

Then inspect the history with:

```bash
git log --oneline
```


In [ ]:
result = subprocess.run(
    ["git", "commit", "-m", "Update addition example"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(result.stdout)
print(subprocess.run(["git", "log", "--oneline", "--decorate", "-5"], cwd=WORKDIR, text=True,
                     capture_output=True).stdout)


# 6. Branches: work on a feature without disturbing `main`

Imagine we want to add subtraction.

Create a branch, implement the feature, test it, and commit it.

The key idea is not the command itself; it is the isolation of work.


In [ ]:
result = subprocess.run(
    ["git", "switch", "-c", "feature/subtraction"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(result.stdout, result.stderr)


In [ ]:
# Add subtraction to the module.
h = WORKDIR / "src" / "math_utils.h"
c = WORKDIR / "src" / "math_utils.c"
main = WORKDIR / "src" / "main.c"

h.write_text(h.read_text().replace("int multiply(int a, int b);", "int multiply(int a, int b);\nint subtract(int a, int b);"))
c.write_text(c.read_text() + "\nint subtract(int a, int b)\n{\n    return a - b;\n}\n")
main.write_text(main.read_text().replace(
    'print_result("10 + 7", add(10, 7));',
    'print_result("10 + 7", add(10, 7));\n    print_result("10 - 7", subtract(10, 7));'
))

run_make()


In [ ]:
subprocess.run(["git", "add", "src"], cwd=WORKDIR)
result = subprocess.run(["git", "commit", "-m", "Add subtraction operation"],
                        cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)


### Merge it

Switch back to `main`, merge the feature branch, and inspect the graph.

Before running the cells, predict:
- Which branch contains the subtraction commit?
- Will `main` contain it before the merge?
- What should `git log --graph --oneline --all` show after the merge?


In [ ]:
for command in [
    ["git", "switch", "main"],
    ["git", "merge", "--no-ff", "feature/subtraction", "-m", "Merge subtraction feature"],
]:
    result = subprocess.run(command, cwd=WORKDIR, text=True, capture_output=True)
    print(result.stdout, result.stderr)

print(subprocess.run(["git", "log", "--graph", "--oneline", "--all"], cwd=WORKDIR, text=True,
                     capture_output=True).stdout)


### Exercise: Branch reasoning

Why is a branch useful if the files are still in the same working directory? Explain what changes when you switch branches, and inspect `git log --oneline --graph --all`.

In [ ]:
result = subprocess.run(["git", "log", "--oneline", "--graph", "--all"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)

### Check your work

In [ ]:
assert "*" in result.stdout
print("Branches are pointers to commits; switching changes which committed snapshot is checked out.")

# 7. A controlled merge conflict

Conflicts are easier to understand if we create one intentionally.

Two branches will make incompatible edits to the same line. Your task is to inspect the conflict and resolve it.


In [ ]:
subprocess.run(["git", "switch", "-c", "feature/format"], cwd=WORKDIR, text=True, capture_output=True)

path = WORKDIR / "src" / "string_utils.c"
text = path.read_text().replace('printf("%s = %d\n", operation, value);',
                                'printf("[RESULT] %s = %d\n", operation, value);')
path.write_text(text)
subprocess.run(["git", "add", "src/string_utils.c"], cwd=WORKDIR)
subprocess.run(["git", "commit", "-m", "Add result prefix"], cwd=WORKDIR, text=True, capture_output=True)

subprocess.run(["git", "switch", "main"], cwd=WORKDIR, text=True, capture_output=True)
path = WORKDIR / "src" / "string_utils.c"
text = path.read_text().replace('printf("%s = %d\n", operation, value);',
                                'printf("RESULT: %s = %d\n", operation, value);')
path.write_text(text)
subprocess.run(["git", "add", "src/string_utils.c"], cwd=WORKDIR)
subprocess.run(["git", "commit", "-m", "Change result formatting"], cwd=WORKDIR, text=True, capture_output=True)

result = subprocess.run(["git", "merge", "feature/format"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)


### Resolve the conflict

Open `src/string_utils.c`.

You should see conflict markers:

```text
<<<<<<<
...
=======
...
>>>>>>>
```

Decide which output format you want, remove **all** conflict markers, save the file, compile the project, and then complete the merge.

This is a good moment to emphasize the workflow:

```text
conflict → inspect → edit → compile/test → git add → git commit
```


In [ ]:
conflicted = WORKDIR / "src" / "string_utils.c"
print(conflicted.read_text())


### Your turn

Resolve the file manually, then run the following verification cell.


In [ ]:
# This cell checks that conflict markers are gone and the project still builds.
text = (WORKDIR / "src" / "string_utils.c").read_text()
assert "<<<<<<<" not in text and "=======" not in text and ">>>>>>>" not in text

result = run_make()
assert result.returncode == 0

subprocess.run(["git", "add", "src/string_utils.c"], cwd=WORKDIR)
commit = subprocess.run(["git", "commit", "-m", "Resolve formatting conflict"],
                        cwd=WORKDIR, text=True, capture_output=True)
print(commit.stdout)


### Exercise: Conflict resolution test

After resolving the conflict, do not just compile. Add a test that proves **both intended behaviors** survived the merge. This is the practical reason tests help with merges.

In [ ]:
test_file = WORKDIR / "test_merge.c"
test_file.write_text(r'''#include <assert.h>
#include <string.h>
#include "src/string_utils.h"
int main(void) {
    /* TODO: replace these with assertions for both merged behaviors. */
    assert(1);
    return 0;
}
''')
print(test_file)

### Check your work

In [ ]:
result = subprocess.run(["gcc", "-Wall", "-Wextra", "-std=c11", "-I.", "test_merge.c", "src/string_utils.c", "-o", "test_merge"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stdout, result.stderr)
print("Now replace the placeholder assertion with assertions about the actual merged behavior.")

# 8. Testing: make the computer check your assumptions

Testing is not just a final activity. Tests become valuable when they are:
- cheap to run;
- repeatable;
- specific about expected behavior;
- run frequently.

We will start with a tiny Python test harness that executes our compiled C program.

Then we will add tests for the C functions themselves.


In [ ]:
# Run the current program as a black-box test.
result = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True)
print(result.stdout)
assert result.returncode == 0
assert "10 + 7" in result.stdout


### Exercise: Design a test before coding it

Write down 3 inputs that could expose a bug in `add`, `multiply`, or formatting. At least one should be a boundary or unusual case.

In [ ]:
# Test ideas:
# 1.
# 2.
# 3.

### Check your work

In [ ]:
print("Good test design starts from behavior and edge cases, not from the implementation.")

## 8.1 Write a regression test

A regression test protects behavior that already worked.

**Exercise:** add an assertion for subtraction. Then intentionally break `subtract()` and observe the test fail.


In [ ]:
output = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True).stdout
assert "10 - 7 = 3" in output, "Subtraction regression test failed"
print("Regression test passed.")


### Exercise: Make the regression test fail

Intentionally introduce a small bug in the implementation so your new regression assertion fails. Observe the failure, then fix the bug and rerun the test.

In [ ]:
# TODO: edit one implementation line, run the test, then restore the correct implementation.

### Check your work

In [ ]:
print("A test that never fails during development is worth less than a test whose failure you have understood.")

## 8.2 Unit testing a C module

A unit test should ideally test a small component independently.

One simple technique in C is to compile a dedicated test executable that links the module being tested with a test `main`.


In [ ]:
test_c = WORKDIR / "test_math_utils.c"
test_c.write_text(r'''
#include <assert.h>
#include "src/math_utils.h"

int main(void)
{
    assert(add(2, 3) == 5);
    assert(add(-2, 2) == 0);
    assert(multiply(4, 5) == 20);
    assert(subtract(10, 7) == 3);
    return 0;
}
'''.lstrip())

result = subprocess.run(
    ["gcc", "-Wall", "-Wextra", "-std=c11", str(test_c), str(WORKDIR / "src" / "math_utils.c"),
     "-o", str(WORKDIR / "test_math_utils")],
    cwd=WORKDIR, text=True, capture_output=True
)
print(result.stderr)
assert result.returncode == 0

test_run = subprocess.run([str(WORKDIR / "test_math_utils")], cwd=WORKDIR, text=True, capture_output=True)
print("exit code:", test_run.returncode)
assert test_run.returncode == 0


### Exercise: Complete the unit test

Extend the unit test with at least 4 assertions covering normal inputs, zero, negative values, and a second operation.

In [ ]:
# TODO: edit test_math_utils.c and add assertions.
print((WORKDIR / "test_math_utils.c").read_text())

### Check your work

In [ ]:
result = subprocess.run(["gcc", "-Wall", "-Wextra", "-std=c11", "-I.", "test_math_utils.c", "src/math_utils.c", "-o", "test_math_utils"], cwd=WORKDIR, text=True, capture_output=True)
print(result.stderr)
if result.returncode == 0:
    run = subprocess.run([str(WORKDIR / "test_math_utils")], cwd=WORKDIR, text=True, capture_output=True)
    print("return code:", run.returncode)
    assert run.returncode == 0

### Your turn — add a test that catches a bug

Add an assertion for a case that is **not already covered**, such as:

```c
assert(multiply(0, 100) == 0);
```

Then temporarily introduce a bug in `multiply()` and verify that the test catches it.

**Key idea:** a test is useful only if it can fail when the implementation is wrong.


# 9. Regression testing and a golden output

Sometimes the easiest specification is an expected output.

We can save a known-good output as a **golden file**, run the program, and compare the new output against it.


In [ ]:
golden = WORKDIR / "expected_output.txt"
output = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True).stdout
golden.write_text(output)

print("Golden output:")
print(golden.read_text())


In [ ]:
actual = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True).stdout
expected = golden.read_text()
assert actual == expected, "Regression detected: program output changed!"
print("Golden-output regression test passed.")


### Exercise: Break the golden output

Change one output string in the application. Run the golden comparison and inspect the diff. Then decide whether the test or the application should change.

In [ ]:
# TODO: change one output string, then run the golden comparison.

### Check your work

In [ ]:
actual = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True).stdout
expected = (WORKDIR / "expected_output.txt").read_text()
print("MATCH" if actual == expected else "DIFFERENT")
if actual != expected:
    import difflib
    print("".join(difflib.unified_diff(expected.splitlines(True), actual.splitlines(True), fromfile="expected", tofile="actual")))

### Discussion

What are the strengths and weaknesses of golden-file tests?

- They are excellent at detecting unexpected output changes.
- They can become noisy when output intentionally changes.
- A good regression suite usually combines them with focused unit/integration tests.


# 10. Integration testing

Unit tests check components. Integration tests check that components work together.

Here, the integration boundary is:

```text
main → math_utils → result → string_utils → stdout
```

Our black-box test exercises that whole path.


In [ ]:
# Integration test: exercise the complete executable.
result = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True)

assert result.returncode == 0
assert "10 + 7" in result.stdout
assert "10 - 7" in result.stdout
print("Integration test passed.")


### Exercise: Find an integration bug

Break the interaction between two modules without breaking either module’s isolated compilation. Then make the integration test catch it.

In [ ]:
# TODO: introduce a bug at the module boundary, run the integration test, and restore it afterward.

### Check your work

In [ ]:
print("The point is to create a failure that a unit test of only one module could miss.")

# 11. Performance testing: measure before optimizing

Performance testing asks a quantitative question.

Instead of saying “this feels slow”, measure elapsed time. For a tiny program the numbers are not interesting, but the workflow is.


In [ ]:
import time

start = time.perf_counter()
for _ in range(100):
    subprocess.run([str(WORKDIR / "calculator")], stdout=subprocess.DEVNULL, check=True)
elapsed = time.perf_counter() - start

print(f"100 executions: {elapsed:.4f} seconds")
print(f"Average: {elapsed / 100:.6f} seconds")


### Exercise: Measure variability

Run the performance measurement 5 times. Calculate the minimum, maximum, and mean runtime. Why is one measurement insufficient?

In [ ]:
import statistics, time
times = []
for _ in range(5):
    start = time.perf_counter()
    subprocess.run([str(WORKDIR / "calculator")], stdout=subprocess.DEVNULL, check=True)
    times.append(time.perf_counter() - start)
print("times:", times)
print("min:", min(times), "max:", max(times), "mean:", statistics.mean(times))

### Check your work

In [ ]:
assert len(times) == 5
print("Variation can come from OS scheduling, caching, background processes, and measurement overhead.")

### Exercise

Modify the program so that it performs substantially more work, then measure again.

The point is not to win a benchmark; it is to practice:

```text
hypothesis → measurement → change → measurement
```


# 12. Logging

`printf()` is useful for user-facing output. Logging is useful for understanding what a program is doing while it runs.

Typical levels include:
- `DEBUG` — detailed diagnostic information;
- `INFO` — normal operational events;
- `WARNING` — something unexpected but recoverable;
- `ERROR` — an operation failed;
- `CRITICAL` — severe failure.

Let's first use Python's logging module to see the idea.


In [ ]:
import logging

logger = logging.getLogger("lecture")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    logger.addHandler(handler)

logger.debug("Detailed state")
logger.info("Program started")
logger.warning("Input is unusual")
logger.error("An operation failed")


### Exercise: Turn a vague print into useful logging

Write three log messages for a hypothetical function: entering the function, successfully loading a file, and failing to parse the file. Choose appropriate levels.

In [ ]:
# logger.debug(...)
# logger.info(...)
# logger.error(...)

### Check your work

In [ ]:
print("Typical choice: DEBUG for detailed flow, INFO for meaningful progress, ERROR for failures that prevent the operation.")

### Your turn — choose the right level

For each event, choose `DEBUG`, `INFO`, `WARNING`, or `ERROR`:

1. “Loaded configuration file.”
2. “User entered an invalid value, so we ask again.”
3. “Could not open the database.”
4. “Current cache size is 3812 entries.”

There is not always one perfect answer; the important question is **what should an operator do with the message?**


## 12.1 Add logging to the C program

C does not have a standard logging framework comparable to Python's `logging`, but a project can define a small logging interface.

We will add a minimal logger that writes levels and messages to `stderr`.


In [ ]:
(WORKDIR / "src" / "logger.h").write_text(r'''
#ifndef LOGGER_H
#define LOGGER_H

void log_info(const char *message);
void log_warning(const char *message);
void log_error(const char *message);

#endif
'''.lstrip())

(WORKDIR / "src" / "logger.c").write_text(r'''
#include <stdio.h>
#include "logger.h"

void log_info(const char *message)
{
    fprintf(stderr, "INFO: %s\n", message);
}

void log_warning(const char *message)
{
    fprintf(stderr, "WARNING: %s\n", message);
}

void log_error(const char *message)
{
    fprintf(stderr, "ERROR: %s\n", message);
}
'''.lstrip())

# Add a simple startup log to main.c.
path = WORKDIR / "src" / "main.c"
text = path.read_text()
text = text.replace('#include "math_utils.h"', '#include "math_utils.h"\n#include "logger.h"')
text = text.replace('int main(void)\n{', 'int main(void)\n{\n    log_info("calculator started");')
path.write_text(text)

# Add logger.o to the automatic Makefile.
make_text = makefile.read_text()
make_text = make_text.replace(
    "OBJECTS = main.o math_utils.o string_utils.o",
    "OBJECTS = main.o math_utils.o string_utils.o logger.o"
)
makefile.write_text(make_text)

run_make()


### Observe stdout vs stderr

Run the program while capturing the two streams separately.

Why is logging often sent to `stderr` rather than mixed with the program's normal output?


In [ ]:
result = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)


### Exercise: Logging vs program output

Modify the C program so a diagnostic message goes to `stderr` while the normal result remains on `stdout`. Capture both streams separately.

In [ ]:
# TODO: edit the logger implementation and run the program.

### Check your work

In [ ]:
result = subprocess.run([str(WORKDIR / "calculator")], text=True, capture_output=True)
print("STDOUT:", repr(result.stdout))
print("STDERR:", repr(result.stderr))
print("Verify that diagnostics and user-facing output are separated.")

# 13. CI/CD: combine the pieces

Continuous Integration (CI) means that changes are automatically built and tested.

We can simulate a CI pipeline in this notebook:

1. clean the project;
2. build from scratch;
3. run unit tests;
4. run integration tests;
5. fail if anything fails.

That is already a useful CI mindset, even before introducing GitHub Actions/Jenkins/etc.


In [ ]:
def run_checked(command, cwd=WORKDIR):
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    print("$", " ".join(map(str, command)))
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

# CI-style clean build
run_checked(["make", "clean"])
run_checked(["make"])

# Unit test
run_checked([str(WORKDIR / "test_math_utils")])

# Rebuild the test in case the source changed.
run_checked([
    "gcc", "-Wall", "-Wextra", "-std=c11",
    str(WORKDIR / "test_math_utils.c"),
    str(WORKDIR / "src" / "math_utils.c"),
    "-o", str(WORKDIR / "test_math_utils")
])
run_checked([str(WORKDIR / "test_math_utils")])

# Integration test
result = run_checked([str(WORKDIR / "calculator")])
assert "10 - 7 = 3" in result.stdout
print("ALL CI CHECKS PASSED")


### Exercise: Build a tiny CI gate

Create a Python function that returns success only if both `make clean && make` and the unit test pass. This is the core idea behind a CI pipeline.

In [ ]:
def ci_gate():
    # TODO: run the build and unit test. Return True only if both succeed.
    return False

print("CI gate result:", ci_gate())

### Check your work

In [ ]:
print("A useful CI gate should fail when the project does not build or its tests fail.")

# 14. Project lifecycle: put the tools together

A large-scale project is not “write code, then submit it”.

A simplified lifecycle is:

```text
requirements
    ↓
design
    ↓
implementation
    ↓
build
    ↓
test
    ↓
review / integration
    ↓
release
    ↓
monitor / maintain
    ↺
```

The tools we practiced support different parts:

| Need | Tool / practice |
|---|---|
| Modular implementation | C modules + headers |
| Repeatable builds | Make |
| Change history | Git |
| Isolated feature work | Branches |
| Collaboration | Merge / pull request |
| Correctness | Unit + integration + regression tests |
| Reliability | CI |
| Diagnosis | Logging |


### Exercise: Choose the right tool

For each scenario, choose the most relevant tool: Make, Git, unit tests, integration tests, logging, or CI.

1. "Only the affected `.o` files should rebuild."
2. "I need to see what changed between two versions."
3. "A helper function returns the wrong value."
4. "Two modules work alone but fail together."
5. "The program behaves incorrectly on a server and I need evidence of what happened."
6. "Every pushed change should be built and tested."

In [ ]:
# Answers:
# 1.
# 2.
# 3.
# 4.
# 5.
# 6.

### Check your work

In [ ]:
print("Expected: 1 Make, 2 Git, 3 unit test, 4 integration test, 5 logging, 6 CI.")

# 15. Final hands-on challenge

You now own the project. Work in pairs or individually.

### Challenge

Add a new `divide()` operation with the following requirements:

1. `divide(int a, int b)` belongs in `math_utils`.
2. Division by zero must be handled safely.
3. Add at least **two unit tests**.
4. Add an integration test through `main`.
5. Update the Make build correctly.
6. Add a useful log message for division-by-zero.
7. Create a Git branch named `feature/division`.
8. Make at least one commit on that branch.
9. Merge it into `main`.
10. Run the full CI-style checks after the merge.

### Suggested workflow

```text
branch
  ↓
implement
  ↓
unit test
  ↓
make
  ↓
integration test
  ↓
commit
  ↓
merge
  ↓
full test suite
```

Do not start by writing everything. Make one small change at a time and keep the project working.


## 15.1 Progressive challenge ladder

Before the final challenge, work through these increasingly independent tasks. **Do not look at the checker until you have attempted each one.**

### Exercise: Level 1 — add a function

Add a `square(int x)` function to the math module. Update the header, implementation, main program, and build. Do not modify the Makefile if your automatic dependencies are working.

In [ ]:
h = WORKDIR / "src" / "math_utils.h"
c = WORKDIR / "src" / "math_utils.c"
# TODO: edit h and c, then update main.c.

### Check your work

In [ ]:
print("Check: header declaration + implementation + a call site, then make should rebuild what is necessary.")

### Exercise: Level 2 — add a unit test

Add a unit test for your new function. Make the test fail once by choosing an incorrect expected value, then correct it.

In [ ]:
# TODO: edit test_math_utils.c and add a square() assertion.

### Check your work

In [ ]:
print("The important part is experiencing both the failing and passing states.")

### Exercise: Level 3 — commit safely

Create a feature branch, commit the implementation and test separately, and inspect the history graph.

In [ ]:
# TODO: use git commands from Python/subprocess.

### Check your work

In [ ]:
print("Aim for small commits with one coherent purpose each.")

### Exercise: Level 4 — break and diagnose

Introduce a deliberate bug. Use the test suite, `git diff`, and logging to determine what changed and where the failure originates.

In [ ]:
# TODO: introduce a bug, then diagnose it using at least two different tools.

### Check your work

In [ ]:
print("A good debugging workflow combines reproducibility, tests, diffs, and targeted observations.")

In [ ]:
# Optional starting point for the challenge:
# Uncomment and complete this TODO yourself.

# h = WORKDIR / "src" / "math_utils.h"
# text = h.read_text()
# text = text.replace("int subtract(int a, int b);", "int subtract(int a, int b);\nint divide(int a, int b);")
# h.write_text(text)
#
# Then implement divide() in math_utils.c, add tests, and update main.c.


# 16. Takeaways

### C modules
- `.h` files define interfaces through declarations.
- `.c` files contain implementations.
- Each translation unit is compiled separately.
- Linking combines object files into an executable.

### Make
- A target depends on prerequisites.
- Make uses timestamps to decide what is stale.
- Header dependencies matter because headers affect compilation.
- Automatic dependency generation scales better than manually maintaining every header dependency.

### Git
- `status` tells you what changed.
- `diff` tells you how it changed.
- staging lets you choose what belongs in a commit.
- branches isolate work.
- merges integrate work.
- conflicts are resolved by editing, then rebuilding/testing.

### Testing
- Unit tests isolate components.
- Integration tests exercise interactions.
- Regression tests protect previously working behavior.
- Performance tests measure behavior quantitatively.
- CI makes the checks repeatable.

### Logging
- Logs are for diagnosis and operational visibility.
- Levels communicate severity and intent.
- Keep normal program output separate from diagnostic output when appropriate.

**The common theme:** automate anything you would otherwise have to remember to do correctly.
